In [1]:
from sedona.spark import SedonaContext
from pyspark.sql import SparkSession

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1794925904908_0003,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
## LOAD FUNCTIONS
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import ArrayType, DoubleType

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
## BRONZE FILES
TRIPS_PATH   = "s3://workshop-sedona-datalake/bronze/porto_trips_parsed.parquet/"
PLACES_PATH = "s3://workshop-sedona-datalake/bronze/porto_places_wkt.parquet"

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
## LOAD DATA INTO DF
df_trips = spark.read.parquet(TRIPS_PATH)
df_trips.createOrReplaceTempView("trips_raw")
df_places = spark.read.parquet(PLACES_PATH)
df_places.createOrReplaceTempView("places_raw")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
### ---- PLACES ###

In [ ]:
## Load OSM places (polygons) ----
places = spark.sql(f"""
SELECT
    place_id,
    place_name,
    place_type,                      -- parish | park | town | poi ...
    ST_GeomFromWKT(geometry_wkt) AS geom
FROM places_raw
WHERE geometry_wkt IS NOT NULL
""")
places.createOrReplaceTempView("places")

In [ ]:
## Print common places
print(f"Total places: {places.count()}")
places.groupBy("place_type").count().show()

In [5]:
df_places_silver = spark.sql(f"""
SELECT
    place_id,
    CASE 
        WHEN place_name = 'unknown' THEN CONCAT(place_type, ' - ', place_id)
            ELSE place_name 
        END AS place_name,
    place_type,                      -- parish | park | town | poi ...
    ST_GeomFromWKT(geometry_wkt) AS geom
FROM places_raw
WHERE geometry_wkt IS NOT NULL
    AND place_type not in ('8', '7')     -- neighborhoods and citis out.
""")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
df_places_silver.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+----------------+-----------+--------------------+
|place_id|      place_name| place_type|                geom|
+--------+----------------+-----------+--------------------+
|      67|   cemetery - 67|   cemetery|POLYGON ((-8.6306...|
|      68| industrial - 68| industrial|POLYGON ((-8.6503...|
|      69|   cemetery - 69|   cemetery|POLYGON ((-8.6470...|
|      70|   cemetery - 70|   cemetery|POLYGON ((-8.5952...|
|      71|     forest - 71|     forest|POLYGON ((-8.6635...|
|      72|     forest - 72|     forest|POLYGON ((-8.6760...|
|      73|   farmland - 73|   farmland|POLYGON ((-8.6787...|
|      74|   farmland - 74|   farmland|POLYGON ((-8.6739...|
|      75|   farmland - 75|   farmland|POLYGON ((-8.6763...|
|      76|   farmland - 76|   farmland|POLYGON ((-8.6770...|
|      77|   farmland - 77|   farmland|POLYGON ((-8.6783...|
|      78|     forest - 78|     forest|POLYGON ((-8.6756...|
|      79|     forest - 79|     forest|POLYGON ((-8.6787...|
|      80|   farmland - 

In [7]:
### ---- TRIPS ---- ###

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
## Print tips
print(f"Total raw trips: {df_trips.count():,}")
df_trips.show(5, truncate=True)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total raw trips: 1,674,152
+-------------------+--------+----------+---------+--------------------+
|            TRIP_ID| TAXI_ID|   trip_ts|CALL_TYPE|              points|
+-------------------+--------+----------+---------+--------------------+
|1403190600620000571|20000571|1403190600|        C|[[-8.485047, 40.9...|
|1403222177620000903|20000903|1403222177|        A|[[-8.645022, 41.1...|
|1403235920620000007|20000007|1403235920|        C|[[-8.602596, 41.1...|
|1403235910620000451|20000451|1403235910|        C|[[-8.602596, 41.1...|
|1403237750620000184|20000184|1403237750|        B|[[-8.611092, 41.1...|
+-------------------+--------+----------+---------+--------------------+
only showing top 5 rows

In [16]:
df_trip_limit = df_trips.limit(10000)
df_trip_limit.createOrReplaceTempView("trips_limit")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
# ============================================================
# Build REAL route LineStrings from GPS
# ============================================================

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [17]:
df_trip_limit.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------+--------+----------+---------+--------------------+
|            TRIP_ID| TAXI_ID|   trip_ts|CALL_TYPE|              points|
+-------------------+--------+----------+---------+--------------------+
|1403190600620000571|20000571|1403190600|        C|[[-8.485047, 40.9...|
|1403222177620000903|20000903|1403222177|        A|[[-8.645022, 41.1...|
|1403235920620000007|20000007|1403235920|        C|[[-8.602596, 41.1...|
|1403235910620000451|20000451|1403235910|        C|[[-8.602596, 41.1...|
|1403237750620000184|20000184|1403237750|        B|[[-8.611092, 41.1...|
|1403233081620000503|20000503|1403233081|        A|[[-8.685657, 41.1...|
|1403235404620000140|20000140|1403235404|        C|[[-8.602533, 41.1...|
|1403235644620000670|20000670|1403235644|        C|[[-8.602506, 41.1...|
|1403234948620000651|20000651|1403234948|        B|[[-8.589258, 41.1...|
|1403235045620000600|20000600|1403235045|        C|[[-8.60256, 41.15...|
|1403234382620000601|20000601|1403234382|        C|

In [19]:
## Build the actual route geometry from the point array ---- ##
## transform each [lon,lat] → "lon lat", join with commas → WKT LINESTRING
routes_line = spark.sql("""
SELECT
    TRIP_ID,
    TAXI_ID,
    CALL_TYPE,
    size(points) AS num_points,
    ST_MakeLine(
        transform(points, p -> ST_Point(p[0], p[1]))
    ) AS geom
FROM trips_limit
""")
routes_line.createOrReplaceTempView("routes_line")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [20]:
print(f"Valid routes: {routes_line.count():,}")
routes_line.show(5, truncate=True)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Valid routes: 10,000
+-------------------+--------+---------+----------+--------------------+
|            TRIP_ID| TAXI_ID|CALL_TYPE|num_points|                geom|
+-------------------+--------+---------+----------+--------------------+
|1403190600620000571|20000571|        C|       226|LINESTRING (-8.48...|
|1403222177620000903|20000903|        A|        26|LINESTRING (-8.64...|
|1403235920620000007|20000007|        C|        11|LINESTRING (-8.60...|
|1403235910620000451|20000451|        C|        16|LINESTRING (-8.60...|
|1403237750620000184|20000184|        B|        10|LINESTRING (-8.61...|
+-------------------+--------+---------+----------+--------------------+
only showing top 5 rows

In [ ]:
## ST_LengthSpheroid -> It measures how long a line (or the perimeter of a polygon) is, in meters, along the curved surface of the Earth.
## For a taxi route, it tells you how many meters the car actually traveled along that pat 

In [ ]:
## curved surface (a spheroid/ellipsoid model of Earth — WGS84)
## Keeps only routes that are ≥ 10 meters long.

In [21]:
## Filter invalid geometries + measure real route length ----
routes_clean = spark.sql("""
    SELECT *,
        ST_LengthSpheroid(geom) AS length_meters
    FROM routes_line
    WHERE ST_IsValid(geom)
      AND ST_LengthSpheroid(geom) >= 10   -- drop <10m noise routes
""")

routes_clean.createOrReplaceTempView("routes_clean")
print(f"Valid routes: {routes_clean.count():,}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Valid routes: 9,966

In [ ]:
## ---- EXPLORATION ONLY
## ----

In [ ]:
#It applies the Douglas-Peucker simplification algorithm — removing points from a line that don't significantly change its shape.
#
#BEFORE (12 points)              AFTER (4 points)
#●─●─●─●                         ●───────●
#       ╲                                 ╲
#        ●─●                               ╲
#           ╲                               ●───────●
#            ●─●─●─●                  
#                   ●─●              
#0.0001 degrees ≈ 11 meters (near the equator; slightly less at Porto's latitude).

In [22]:
## Simplify routes at scale (the 15 TB → 1 TB story) ----
## 0.0001 deg ≈ 11m tolerance — reduces vertices while keeping the path shape
routes_simplified = spark.sql("""
SELECT
    TRIP_ID,
    TAXI_ID,
    length_meters,
    ST_NPoints(geom)                           AS points_before,
    ST_SimplifyPreserveTopology(geom, 0.0001)  AS geom,
    ST_NPoints(ST_SimplifyPreserveTopology(geom, 0.0001)) AS points_after
FROM routes_clean
""")
routes_simplified.createOrReplaceTempView("routes_simplified")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Valid routes: 9,966

In [23]:
## COMPARE ROUTES SIMPLIFICATION
spark.sql("""
    SELECT
        SUM(points_before) AS total_before,
        SUM(points_after)  AS total_after,
        ROUND(100.0*(1 - SUM(points_after)/SUM(points_before)),1) AS pct_reduction
    FROM routes_simplified
""").show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+-----------+-------------+
|total_before|total_after|pct_reduction|
+------------+-----------+-------------+
|      494834|     269341|         45.6|
+------------+-----------+-------------+

In [ ]:
## ---- 
## ---- END EXPLORATION ONLY

In [ ]:
## TRANSFORM SILVER LAYER

In [24]:
df_routes_simplified = spark.sql("""
SELECT
    TRIP_ID,
    TAXI_ID,
    length_meters,
    ST_SimplifyPreserveTopology(geom, 0.0001)  AS geom,
    ST_NPoints(ST_SimplifyPreserveTopology(geom, 0.0001)) AS points
FROM routes_clean
""")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [25]:
df_routes_simplified.show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------+--------+------------------+--------------------+------+
|            TRIP_ID| TAXI_ID|     length_meters|                geom|points|
+-------------------+--------+------------------+--------------------+------+
|1403190600620000571|20000571|   42317.505294946|LINESTRING (-8.48...|   162|
|1403222177620000903|20000903| 3381.052178675152|LINESTRING (-8.64...|    23|
|1403235920620000007|20000007| 766.0601247749518|LINESTRING (-8.60...|     8|
|1403235910620000451|20000451| 1049.484550942543|LINESTRING (-8.60...|     8|
|1403237750620000184|20000184| 1197.384005156106|LINESTRING (-8.61...|     7|
|1403233081620000503|20000503|1760.8142220620477|LINESTRING (-8.68...|     6|
|1403235404620000140|20000140| 3108.707714355438|LINESTRING (-8.60...|    12|
|1403235644620000670|20000670| 2890.599725861061|LINESTRING (-8.60...|    12|
|1403234948620000651|20000651|2075.7385583948117|LINESTRING (-8.58...|     6|
|1403235045620000600|20000600| 6425.287006664203|LINESTRING (-8.

In [ ]:
## SAVE SILVER LAYER

In [27]:
df_routes_simplified.write.format("geoparquet").mode("overwrite").save("s3://workshop-sedona-datalake/silver/routes_simplified.geoparquet/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
## SAVE PLACES

In [28]:
df_places_silver.write.format("parquet").mode("overwrite").save("s3://workshop-sedona-datalake/silver/places.parquet/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…